# AudioShuttle — Voice-Controlled Professional Music Production

**Gemma 4 Good Hackathon — Digital Equity & Inclusivity Track**

> **TL;DR:** AudioShuttle uses Google's Gemma 4 E4B model as a domain expert to translate natural language voice commands into complete professional DAW arrangements. For the 250M+ people with motor impairments who cannot use traditional mouse/keyboard workflows, this is the first hands-free professional music production tool.

---

## The Problem

Professional audio production is **mouse-and-keyboard only**. Every DAW operation requires physical interaction that **250 million+ people with motor impairments cannot perform**:
- Cerebral palsy, ALS, multiple sclerosis, arthritis
- Limb differences, spinal cord injuries, RSI
- Any condition affecting fine motor control

**No existing solution provides full professional music production capability via voice.**

---

## The Solution: AudioShuttle + Gemma 4 E4B

AudioShuttle bridges natural language to REAPER using Gemma 4 E4B as a domain expert translator:

1. **User speaks/types** a music command: *"create a rock project called Midnight Drive at 130 BPM"*
2. **Gemma 4 E4B translates** using 81K context for genre profiles + tool-use for multi-step orchestration
3. **OSC bridge executes** 40+ DAW operations in sequence with state verification
4. **REAPER produces** a complete arrangement with 8 tracks, 9 markers, routing, and FX

**Why Gemma 4 E4B is essential:**
- 81K context fits entire genre profile + tool schema in one prompt
- Tool-use enables reasoning about multi-step DAW orchestration
- Domain expertise understands music concepts, not just raw text


## Architecture

```
┌──────────────────────────────────────────────────┐
│  User (Voice or Text)                            │
│  "create a rock project called Midnight Drive"   │
└──────────────────────┬───────────────────────────┘
                        │
                        ▼
┌──────────────────────────────────────────────────┐
│  AudioShuttle Web UI (http://localhost:8765)     │
│                                                  │
│  Gemma 4 E4B — Domain Expert Translator          │
│  - 81K context for genre profiles + tools        │
│  - Tool-use for 9-step DAW orchestration         │
│  - 11 genres, 8 instruments, 7 FX chain types    │
└──────────────────────┬───────────────────────────┘
                        │ OSC (localhost:8000)
                        ▼
┌──────────────────────────────────────────────────┐
│  REAPER 7.71 (Linux)                            │
│  - Lua watcher for insert, MIDI, sends           │
│  - Full arrangement, routing, FX                 │
└──────────────────────────────────────────────────┘
```

**Docker Stack:** GPU-accelerated Gemma 4 E4B (AMD ROCm, RX 6950 XT, 99 layers) + AudioShuttle server


## Verified End-to-End Pipeline

The full pipeline has been tested and verified working:

```
Step 1: NL command
  "create a rock project called Midnight Drive at 130 bpm"

Step 2: Gemma 4 E4B translation
  create_genre_project(genre="rock", modifiers={tempo: 130, name: "Midnight Drive"})

Step 3: 9-step DAW pipeline executes:
  - Set tempo to 130 BPM
  - Create 9 song markers (intro -> verse -> chorus -> verse -> chorus -> solo -> chorus -> outro)
  - Insert 8 tracks (Keys, Lead Guitar, Bass, Vocals, Rhythm Guitar, Drums, Guitars Bus, Submaster)
  - Rename all tracks to correct names
  - Apply genre-appropriate colors
  - Route instruments to bus/submaster sends
  - Apply FX chains to buses

Step 4: Verification
  - 8 tracks, 9 markers, 130 BPM, all colors applied

Total time: ~45 seconds from voice command to complete arrangement
```


## What Makes This "4 Good"

### 1. Digital Equity - Real Accessibility, Not Just Play/Stop

This isn't simple "play/stop via voice" - it's **full professional music production capability**:
- Create complete multi-track arrangements via voice
- Generate genre-appropriate MIDI patterns with section awareness
- Route audio through professional bus/submaster chains with effects
- Set volumes, pans, colors, markers - all via voice

**Impact:** For the first time, a motor-impaired musician can produce a full professional arrangement without touching a keyboard or mouse.

### 2. Speed - Idea to Arrangement in Under 60 Seconds

Traditional workflow: 45+ minutes of DAW setup. AudioShuttle: ~45 seconds.

### 3. Simplicity - Describe Music, Not Software

"I want a metal song with a long atmospheric intro and more guitar solos" -> Gemma 4 E4B interprets:
- Tempo: 160 BPM
- Intro: 16 bars of ambient guitar with reverb
- 2x solo sections (not the typical 1)
- Additional lead guitar density throughout chorus

### 4. Open - AI-Agnostic, MIT-Licensed

AudioShuttle is an open bridge. Gemma 4 E4B is the current implementation, but any tool-capable LLM works. No lock-in.


## Technical Highlights

### Gemma 4 E4B Tool Schema

The model uses 12 tool schemas for DAW operations:

| Tool | Parameters | Description |
|------|------------|-------------|
| `create_genre_project` | genre, name, modifiers | Full project from NL |
| `insert_track` | role | Insert named track |
| `rename_track` | track, name | Rename by index |
| `set_track_volume` | track, value | 0.0-1.0 |
| `set_track_pan` | track, value | -1.0 to +1.0 |
| `set_track_color` | track, hex | Track color |
| `play` / `stop` | - | Transport control |
| `set_tempo` | bpm | BPM setting |

### Multi-Step Orchestration

Gemma 4 E4B reasons about the *order* of operations - tracks must be inserted before they can be renamed, state must be verified between steps, stray tracks must be cleaned up.

### Genre Profile System

11 genres with:
- Tempo ranges
- Instrument families
- Bus routing rules
- FX chain types (reverb sends, compression, EQ)


## Repository and Reproducibility

**GitHub:** https://github.com/carlkrott/audioshuttle

### Quick Start
```bash
# 1. Get model files (~5.7GB)
# gemma-4-E4B-it-UD-Q4_K_XL.gguf + gemma-4-e4b-mmproj-BF16.gguf

# 2. Start stack
docker compose up -d

# 3. Configure REAPER (OSC on 8000/9000, load __startup.lua)

# 4. Create a project
curl "http://localhost:8765/replay?cmd=create+a+rock+project+called+Test+at+120+bpm"
```

### Complete E2E Test (Verified)

See the repository README for complete E2E verification showing 8 tracks + 9 markers + 130 BPM created from a single NL command.

---

*Built for the accessibility community - because music is for everyone.*

**Team:** Carl Krott
**Competition:** [Kaggle Gemma 4 Good Hackathon](https://kaggle.com/competitions/gemma-4-good-hackathon)
**Track:** Digital Equity and Inclusivity
